# 04 · Embeddings

Turns every kept review into a vector with `BAAI/bge-m3`: one model for all languages, so reviews about the same thing end up close together whatever language they're in. Vectors go to `data/embeddings/<model>/part-*.parquet`.

Re-run after collecting and cleaning more: only new reviews (and reviews edited since) get embedded. You can stop it at any time; finished parts are kept.

Why bge-m3: on 3,000 of our reviews it was the only model compared (vs Qwen3-Embedding-0.6B and multilingual-e5-base) whose clusters mixed languages instead of splitting by language. 

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from src.config import ROOT
from src.db import read

In [ ]:
MODEL = "BAAI/bge-m3"
PROMPT = ""          # text put before every review; bge-m3 needs none
BATCH = 128          # lower it if CUDA runs out of memory
PART_SIZE = 20_000   # reviews per file, also the most a stop can lose

EMB_DIR = ROOT / "data" / "embeddings" / MODEL.replace("/", "__")
EMB_DIR.mkdir(parents=True, exist_ok=True)

## What's left to embed

In [ ]:
parts = sorted(EMB_DIR.glob("part-*.parquet"))
done = pd.concat(
    [pd.read_parquet(p, columns=["recommendation_id", "updated_at"]) for p in parts]
    or [pd.DataFrame({"recommendation_id": [], "updated_at": []})]
).groupby("recommendation_id")["updated_at"].max()

reviews = read.reviews(columns=["recommendation_id", "review", "updated_at"], only_kept=True)
embedded_at = pd.to_datetime(reviews["recommendation_id"].map(done), utc=True)
todo = reviews[embedded_at.isna() | (reviews["updated_at"] > embedded_at)]
print(f"{len(reviews):,} kept reviews, {len(done):,} embedded, {len(todo):,} to do")

## Embed

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
model = SentenceTransformer(MODEL, device=device, model_kwargs={"dtype": dtype})
model.max_seq_length = 512

run = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
with tqdm(total=len(todo), unit="review") as bar:
    for i, start in enumerate(range(0, len(todo), PART_SIZE)):
        part = todo.iloc[start : start + PART_SIZE]
        part = part.iloc[np.argsort(part["review"].str.len().to_numpy())] 
        texts = part["review"].tolist()
        vectors = []
        for b in range(0, len(texts), BATCH):
            batch = texts[b : b + BATCH]
            vectors.append(model.encode(batch, prompt=PROMPT, batch_size=BATCH, normalize_embeddings=True))
            bar.update(len(batch))
        vectors = np.concatenate(vectors).astype(np.float16)
        table = pa.table({
            "recommendation_id": pa.array(part["recommendation_id"]),
            "updated_at": pa.array(part["updated_at"]),
            "embedding": pa.FixedSizeListArray.from_arrays(vectors.ravel(), vectors.shape[1]),
        })
        path = EMB_DIR / f"part-{run}-{i:04d}.parquet"
        pq.write_table(table, path.with_suffix(".tmp"))
        path.with_suffix(".tmp").rename(path)  # a stop mid-write leaves no broken part

## Check: nearest reviews across languages

Each test sentence should find reviews about the same thing, in any language. Reads one file at a time, so memory stays small.

In [ ]:
parts = sorted(EMB_DIR.glob("part-*.parquet"))
meta = pd.concat(
    [pd.read_parquet(p, columns=["recommendation_id", "updated_at"]).assign(part=i) for i, p in enumerate(parts)],
    ignore_index=True,
)
meta["row"] = meta.groupby("part").cumcount()
newest = meta.sort_values("updated_at").drop_duplicates("recommendation_id", keep="last")  # edited review: newest vector
rows_by_part = {i: np.sort(g.to_numpy()) for i, g in newest.groupby("part")["row"]}
print(f"{len(newest):,} reviews in {len(parts)} files")

In [ ]:
PROBES = [
    "Can't connect, the servers are always down.",
    "Не могу подключиться, серверы постоянно лежат.",
    "服务器老是连不上，根本进不去游戏。",
    "Way too expensive for what it is, wait for a sale.",
    "Слишком дорого за такой контент, ждите скидку.",
    "太贵了，不值这个价，等打折再买。",
    "Terrible optimization, constant FPS drops and stuttering.",
    "Ужасная оптимизация, постоянные просадки FPS и фризы.",
    "优化太差，掉帧卡顿严重。",
]
TOP = 5
probes = model.encode(PROBES, prompt=PROMPT, normalize_embeddings=True)
hits = []
for i, path in enumerate(tqdm(parts, unit="file")):
    if i not in rows_by_part:
        continue
    part = pq.read_table(path).take(rows_by_part[i])
    vectors = part["embedding"].combine_chunks().flatten().to_numpy().reshape(len(part), -1)
    sims = vectors.astype(np.float32) @ probes.T
    ids = part["recommendation_id"].to_numpy()
    for j in range(len(PROBES)):
        for k in sims[:, j].argsort()[-TOP:]:
            hits.append((j, sims[k, j], ids[k]))
hits = pd.DataFrame(hits, columns=["probe", "similarity", "recommendation_id"])

info = reviews.set_index("recommendation_id")[["review", "text_language"]]
for j, probe in enumerate(PROBES):
    print(f"\n{probe}")
    for hit in hits[hits["probe"] == j].nlargest(TOP, "similarity").itertuples():
        text, lang = info.loc[hit.recommendation_id] if hit.recommendation_id in info.index else ("", "?")
        print(f"  {hit.similarity:.2f} [{lang}] {text[:100]!r}")